# e5 · Elastic Net vs XGBoost

Each B5 pair matches the macro-variable set and compares **XGBoost (a)** with **Elastic Net (b)**.
Both use previous-month log RV and the matched macro set, with expanding training windows.

| Model | Refitting | Hyperparameter retuning |
|---|---|---|
| XGBoost | Monthly | Every 24 months |
| Elastic Net | Monthly | Monthly |

Results compare the forecasting procedures, including their different tuning schedules.

| Pair | Macro information | a (XGBoost) | b (Elastic Net) |
|---|---|---|---|
| B5.1 | No macro | `xgb_no_macro` | `en_no_macro` |
| B5.2 | Macro without VIX | `xgb_no_vix` | `en_no_vix` |
| B5.3 | VIX only | `xgb_vix_only` | `en_vix_only` |
| B5.4 | Full macro + VIX | `xgb_full_macro` | `en_full_macro` |

**Output** in `forecasting/results/`: `e5_losses.csv`, `e5_evaluation.csv`.
`gain_t > 0` favours XGBoost; `loss_ratio` is XGBoost loss / Elastic Net loss.


In [1]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

from research_config import BLOCKS, TICKERS

FORECAST_DIR = MODELS_DIR / "forecasting" / "forecasts"
EN_PATH      = FORECAST_DIR / "en_forecasts.csv"
XGB_PATH     = FORECAST_DIR / "xgb_forecasts.csv"
RESULTS_DIR  = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Use the same B5 pairs as portfolio evaluation, including no-macro.
block = BLOCKS["B5"]

COLS = ["model", "etf", "date", "actual_logRV", "actual_RV", "fc_log", "fc_level"]
df = pd.concat([pd.read_csv(EN_PATH, parse_dates=["date"])[COLS],
                pd.read_csv(XGB_PATH, parse_dates=["date"])[COLS]], ignore_index=True)
available = set(df["model"])
pairs = [p for p in block["pairs"] if p[1] in available and p[2] in available]
print(f"{len(pairs)} of {len(block['pairs'])} E5 pairs runnable")

3 of 3 E5 pairs runnable


## Evaluation helpers

In [2]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))

def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)

def hac_t(d):
    '''t-statistic of mean(d) with a Newey-West variance.'''
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))

def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Losses use the common ETF and forecast-origin sample across all compared specifications.

In [3]:
models = [m for m in block["models"] if m in available]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]

# the EN and XGB files must record identical realised values for the same (ETF, month)
spread = d.groupby(["etf", "date"])[["actual_logRV", "actual_RV"]].agg(lambda x: x.max() - x.min())
assert spread.max().max() < 1e-9, "EN/XGB realised values differ"

d["MSE_log"]   = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"]     = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}, {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e5_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)

2008-02 to 2025-11, 6420 rows


,QLIKE,MSE_log,MSE_level
model,,,
en_full_macro,0.3247,0.4902,0.0011
en_no_vix,0.3797,0.5371,0.0004
en_vix_only,0.2913,0.4769,0.0015
xgb_full_macro,0.3370,0.4866,0.0001
xgb_no_vix,0.3773,0.5290,0.0001
xgb_vix_only,0.3394,0.5074,0.0001


## Pairwise tests

In [4]:
def one_sided_p(t):
    return float(1 - norm.cdf(t))

loss_idx = losses.set_index(["model", "ETF"])
rows = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"])
    B = d[d["model"] == b].set_index(["date", "etf"])
    diffs = {"QLIKE": A["QLIKE"] - B["QLIKE"],        # negative favours a
             "MSE_log": A["MSE_log"] - B["MSE_log"]}  # negative favours a
    for etf in ["pooled"] + TICKERS:
        for loss, diff in diffs.items():
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            t = -hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": "DM",
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100
tests.to_csv(RESULTS_DIR / "e5_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value
0,E5.1,xgb_full_macro,en_full_macro,QLIKE,DM,1.0378,-0.4910,0.6883
1,E5.1,xgb_full_macro,en_full_macro,MSE_log,DM,0.9928,0.1399,0.4444
2,E5.2,xgb_vix_only,en_vix_only,QLIKE,DM,1.1650,-1.7694,0.9616
3,E5.2,xgb_vix_only,en_vix_only,MSE_log,DM,1.0641,-1.1357,0.8720
4,E5.3,xgb_no_vix,en_no_vix,QLIKE,DM,0.9938,0.1114,0.4556
5,E5.3,xgb_no_vix,en_no_vix,MSE_log,DM,0.9850,0.3783,0.3526


## Sector results

In [5]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def by_sector(loss):
    t = tests[tests["loss"] == loss].copy()
    t["cell"] = [f"{r:.2f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell")[TICKERS + ["pooled"]]
    table.columns = [f"{c} {SECTORS[c]}" for c in TICKERS] + ["Pooled"]
    return table.rename_axis(None)

for loss in ["MSE_log", "QLIKE"]:
    print(f"{loss}: ratio (p-value), DM")
    display(by_sector(loss))

MSE_log: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E5.1: xgb_full_macro vs en_full_macro,1.08 (0.907),0.97 (0.314),0.97 (0.325),0.98 (0.397),0.98 (0.402),0.99 (0.444)
E5.2: xgb_vix_only vs en_vix_only,1.12 (0.988),1.08 (0.908),1.10 (0.885),1.00 (0.511),1.02 (0.580),1.06 (0.872)
E5.3: xgb_no_vix vs en_no_vix,1.03 (0.695),0.97 (0.301),0.97 (0.284),0.97 (0.302),0.99 (0.448),0.99 (0.353)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E5.1: xgb_full_macro vs en_full_macro,1.09 (0.882),0.95 (0.330),1.10 (0.692),0.99 (0.431),1.06 (0.876),1.04 (0.688)
E5.2: xgb_vix_only vs en_vix_only,1.21 (0.996),1.15 (0.970),1.22 (0.813),1.11 (0.982),1.12 (0.973),1.16 (0.962)
E5.3: xgb_no_vix vs en_no_vix,1.04 (0.659),0.96 (0.326),1.05 (0.686),0.89 (0.127),1.04 (0.692),0.99 (0.456)


## Full-information report table

XGBoost with all macro predictors and VIX versus Elastic Net with the same inputs.
Loss changes are calculated from the unrounded ratios as `100 * (XGBoost loss / Elastic Net loss - 1)` and displayed to two decimal places; negative values favour XGBoost. The p-values are one-sided Diebold–Mariano tests.


In [6]:
# Report table: full-information XGBoost versus full-information Elastic Net.
full_info = tests.loc[
    tests["a"].eq("xgb_full_macro")
    & tests["b"].eq("en_full_macro")
    & tests["test"].eq("DM")
].pivot(index="ETF", columns="loss", values=["loss_ratio", "p_value"])
full_info = full_info.loc[TICKERS + ["pooled"]]
full_info_pct = (full_info["loss_ratio"] - 1) * 100

full_info_report_table = pd.DataFrame({
    "log-MSE": full_info_pct["MSE_log"].map(lambda value: f"{value:+.2f}%"),
    "DM p (log-MSE)": full_info["p_value"]["MSE_log"].map(fmt_p),
    "QLIKE": full_info_pct["QLIKE"].map(lambda value: f"{value:+.2f}%"),
    "DM p (QLIKE)": full_info["p_value"]["QLIKE"].map(fmt_p),
}).rename(index={"pooled": "Pooled"}).rename_axis("ETF")
full_info_report_table.columns = pd.MultiIndex.from_arrays([
    ["XGBoost vs Elastic Net"] * 4,
    ["log-MSE", "Diebold-Mariano p-value", "QLIKE", "Diebold-Mariano p-value"],
])
display(full_info_report_table)


XGBoost vs Elastic Net                                                         
                      log-MSE Diebold-Mariano p-value    QLIKE Diebold-Mariano p-value
ETF                                                                                   
XLB                    +3.56%                   0.757   +7.38%                   0.922
XLE                    +0.06%                   0.504   +8.99%                   0.868
XLF                    +5.28%                   0.816  +11.91%                   0.909
XLI                    +1.73%                   0.603   +6.97%                   0.757
XLY                    -3.81%                   0.295   -4.31%                   0.358
Pooled                 +1.43%                   0.642   +6.64%                   0.906